# VectorCypher Retriever

The `VectorCypherRetriever` enhances vector search with custom Cypher graph traversal. After the vector index finds matching chunks, a Cypher query runs on each match to pull in related graph context: neighboring chunks, document metadata, or connected entities.

**Learning Objectives:**
- Write a Cypher retrieval query that traverses from matched chunks to related nodes
- Use `VectorCypherRetriever` to combine semantic search with graph context
- Build a GraphRAG pipeline that grounds LLM answers in both text and graph data

Pure vector search returns isolated chunks. VectorCypher returns the chunk plus whatever the graph knows about its neighborhood.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from lib.data_utils import get_embedder, get_llm
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print(f'Connected to Neo4j!')
print(f'LLM: {llm.model_id}')
print(f'Embedder: {embedder.model_id}')

## Define Retrieval Query

The retrieval query runs on each chunk returned by vector search. It receives the matched `node` and `score` variables and must return `text`, `score`, and optionally `metadata`.

This query traverses from the matched chunk to its source document, then follows the `FILED` relationship to find the company that filed it. It also looks for products extracted from the chunk via `FROM_CHUNK`. This pulls structured knowledge graph data alongside the unstructured chunk text.

In [ ]:
RETRIEVAL_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies,
     collect(DISTINCT risk.name)[0..5] AS risks,
     collect(DISTINCT product.name)[0..5] AS products
RETURN node.text AS text,
       score,
       {document: doc.name, companies: companies, products: products, risks: risks} AS metadata
"""

print('Retrieval query defined!')

## Initialize VectorCypher Retriever

The `VectorCypherRetriever` takes a vector index, an embedder, and a custom Cypher retrieval query. When you search, it embeds the query, finds matching chunks via the vector index, then runs the Cypher query on each match to pull in graph context.

In [ ]:
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=RETRIEVAL_QUERY
)

print('VectorCypherRetriever initialized!')

## GraphRAG with Graph Context

Build a GraphRAG pipeline with the VectorCypher retriever. The LLM now receives the matched chunk along with connected entity data — companies, products, and risk factors — producing answers grounded in both unstructured text and structured knowledge graph relationships.

In [ ]:
rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)

query = "What are the key risk factors mentioned in Apple's 10-K filing?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

print('\n\n=== Enriched Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    content_str = str(item.content)
    preview = content_str[:300] + '...' if len(content_str) > 300 else content_str
    print(f'\n[{i}] {preview}')

## Summary

The `VectorCypherRetriever` adds graph context to every vector search result. The retrieval query traverses `FROM_DOCUMENT` to the source document, then follows the `FILED` relationship to find the company that filed it. This gives the LLM structured context (company names, product lists, risk factor labels) alongside the unstructured chunk, leading to more grounded and specific answers.

In this lab you built a complete GraphRAG pipeline from scratch:

1. **Data Loading** — structured entities and unstructured chunks in a two-layer graph
2. **Embeddings** — 1024-dimensional vectors on every chunk, indexed for similarity search
3. **VectorCypherRetriever** — semantic search enriched with graph traversal for entity context

This is the end of the workshop. You now have hands-on experience building every layer of a GraphRAG application: from raw data to knowledge graph to retrieval-augmented generation.

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')